# SolarScan — 01 · Exploration des données (EDA approfondie)

Détection de défauts sur panneaux solaires par imagerie thermique.

**Un bon Data Scientist ne modélise jamais à l'aveugle.** Avant tout modèle, on interroge les données :

1. Les métadonnées : comment c'est étiqueté ?
2. **Intégrité** : fichiers manquants ? tailles cohérentes ? combien de canaux ?
3. **Distribution des classes** : équilibrée ou non ?
4. Indicateurs de déséquilibre
5. **Intensités de pixels** : en thermique, un défaut = de la chaleur = des pixels clairs
6. **Galerie** d'exemples (variabilité intra-classe)
7. **Image moyenne** par classe (motifs spatiaux)
8. Bilan → décisions de modélisation

> Exécute les cellules une par une (**Maj + Entrée**) et observe chaque sortie.

## 0. Préparation

In [ ]:
# À lancer UNE fois (installe dans le noyau du notebook) :
%pip install -q pandas matplotlib pillow

In [ ]:
import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = Path('../data')   # le notebook est dans notebooks/, les donnees dans data/
print('Dossier de donnees :', DATA_DIR.resolve())
print('Existe :', DATA_DIR.exists())

## 1. Charger les métadonnées

`module_metadata.json` décrit chaque image : son chemin (`image_filepath`) et sa classe (`anomaly_class`).

In [ ]:
with open(DATA_DIR / 'module_metadata.json', encoding='utf-8') as f:
    meta = json.load(f)

print('Nombre entrees :', len(meta))
list(meta.items())[:3]

## 2. Contrôles d'intégrité

Avant d'analyser, on **vérifie la qualité** : autant de fichiers que d'étiquettes ? des images manquantes ? toutes à la même taille ? combien de canaux (gris ou couleur) ?

In [ ]:
img_dir = DATA_DIR / 'images'
files_on_disk = list(img_dir.glob('*.jpg'))
print('Fichiers .jpg sur disque :', len(files_on_disk))
print('Entrees dans le JSON     :', len(meta))

# Chaque entree pointe-t-elle vers un fichier existant ? (echantillon de 2000)
sample_paths = [DATA_DIR / v['image_filepath'] for v in list(meta.values())[:2000]]
missing = [p for p in sample_paths if not p.exists()]
print('Manquants sur 2000 verifies :', len(missing))

# Tailles et modes (canaux) sur un echantillon de 300 images
sample = random.sample(list(meta.values()), 300)
sizes = Counter(Image.open(DATA_DIR / v['image_filepath']).size for v in sample)
modes = Counter(Image.open(DATA_DIR / v['image_filepath']).mode for v in sample)
print('Tailles (largeur, hauteur) :', dict(sizes))
print('Modes / canaux             :', dict(modes))

## 3. Distribution des classes

La question clé en classification : **les classes sont-elles équilibrées ?**

In [ ]:
counts = Counter(v['anomaly_class'] for v in meta.values())
df = (pd.DataFrame(counts.items(), columns=['classe', 'images'])
        .sort_values('images', ascending=False)
        .reset_index(drop=True))
df['part_%'] = (100 * df['images'] / df['images'].sum()).round(1)
df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

ax[0].bar(df['classe'], df['images'], color='#0f3c82')
ax[0].set_xticklabels(df['classe'], rotation=45, ha='right')
ax[0].set_ylabel('Nombre images')
ax[0].set_title('Distribution des 12 classes')

n_normal = counts['No-Anomaly']
n_anom = sum(counts.values()) - n_normal
ax[1].pie([n_normal, n_anom], labels=['No-Anomaly', 'Anomalies'],
          autopct='%1.1f%%', colors=['#9ec9e2', '#0f3c82'])
ax[1].set_title('Sain vs Anomalies')

plt.tight_layout()
plt.show()

## 4. Indicateurs de déséquilibre

Quantifions le déséquilibre et le piège de l'accuracy.

In [ ]:
total = int(df['images'].sum())
maj = df.iloc[0]
mini = df.iloc[-1]
print('Total images        :', total)
print('Classe majoritaire  :', maj['classe'], int(maj['images']))
print('Classe minoritaire  :', mini['classe'], int(mini['images']))
print('Ratio maj / min     :', round(maj['images'] / mini['images'], 1), 'x')
print('Baseline naive (tout-majoritaire) :', round(100 * maj['images'] / total, 1), '%')

## 5. Charger les images en mémoire

On charge **toutes** les images en une fois dans un tableau `X` (et les classes dans `y`). Les images sont minuscules (24×40), donc ça tient en mémoire.

> ⏳ Cette cellule lit 20 000 fichiers : compte **~1 minute**.

In [ ]:
paths = [v['image_filepath'] for v in meta.values()]
y = np.array([v['anomaly_class'] for v in meta.values()])

X = np.stack([np.array(Image.open(DATA_DIR / p).convert('L')) for p in paths])
print('X shape :', X.shape, '| dtype :', X.dtype)   # (20000, 24, 40)
print('y shape :', y.shape)
print('Intensite : min', X.min(), '| max', X.max(), '| moyenne', round(float(X.mean()), 1))

## 6. Analyse des intensités de pixels (le cœur du thermique)

En imagerie thermique, **un défaut chauffe** → il apparaît plus clair. Donc l'intensité des pixels porte de l'information. Regardons si les classes 'chaudes' (Hot-Spot, Offline-Module...) ressortent vraiment.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(X.ravel(), bins=50, color='#0f3c82')
plt.xlabel('Intensite du pixel (0 = froid, 255 = chaud)')
plt.ylabel('Frequence')
plt.title('Distribution des intensites de pixels (toutes images)')
plt.show()

In [ ]:
# Intensite moyenne par classe
mean_int = {c: float(X[y == c].mean()) for c in df['classe']}
mi = pd.Series(mean_int).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
plt.bar(mi.index, mi.values, color='#b5651d')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Intensite moyenne')
plt.title('Intensite thermique moyenne par classe')
plt.tight_layout()
plt.show()
mi.round(1)

In [ ]:
# Distribution de l intensite moyenne PAR IMAGE, classe par classe
order = list(mi.index)
data = []
for c in order:
    sub = X[y == c]
    data.append(sub.reshape(sub.shape[0], -1).mean(axis=1))

plt.figure(figsize=(10, 5))
plt.boxplot(data, showfliers=False)
plt.xticks(range(1, len(order) + 1), order, rotation=45, ha='right')
plt.ylabel('Intensite moyenne par image')
plt.title('Variabilite de l intensite, par classe')
plt.tight_layout()
plt.show()

## 7. Galerie d'exemples (variabilité intra-classe)

Une seule image ne suffit pas : à l'intérieur d'une même classe, les défauts varient. Affichons **4 exemples par classe**, pour les 12 classes.

In [ ]:
classes_all = list(df['classe'])
n_per = 4
fig, axes = plt.subplots(len(classes_all), n_per, figsize=(n_per * 1.5, len(classes_all) * 1.4))
for row, c in enumerate(classes_all):
    idx = np.where(y == c)[0][:n_per]
    for col in range(n_per):
        ax = axes[row, col]
        ax.set_xticks([]); ax.set_yticks([])
        if col < len(idx):
            ax.imshow(X[idx[col]], cmap='inferno')
        if col == 0:
            ax.set_ylabel(c, rotation=0, ha='right', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 8. Image moyenne par classe (motifs spatiaux)

En moyennant toutes les images d'une classe, on révèle **où** la chaleur se concentre typiquement (un coin, une bande, tout le module...).

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for ax, c in zip(axes.ravel(), classes_all):
    ax.imshow(X[y == c].mean(axis=0), cmap='inferno')
    ax.set_title(c, fontsize=8)
    ax.axis('off')
plt.suptitle('Image moyenne par classe')
plt.tight_layout()
plt.show()

## 9. Bilan de l'EDA

**Ce qu'on a appris :**
- 20 000 images thermiques, **24×40 px**, niveaux de gris, 12 classes — données propres (à vérifier dans la section 2).
- **Fort déséquilibre** : No-Anomaly ≈ 50 %, certaines anomalies < 1 % (ratio ~57×).
- **L'intensité moyenne diffère selon les classes** → le signal thermique est réel et exploitable.
- Forte **variabilité intra-classe** → un modèle simple par seuil ne suffira pas, d'où le Deep Learning.

**Décisions de modélisation qui en découlent :**
1. Métrique principale = **macro-F1** (l'accuracy est trompeuse ici).
2. **Pondération des classes** à l'entraînement (ou rééchantillonnage).
3. Découpage train / val / test **stratifié**.
4. **Normalisation** des intensités + augmentation de données.

➡️ Prochain notebook : **préparation des données** (les splits stratifiés), puis la modélisation.